# Tutorial: Searching a STAC catalog for multiple sites

This tutorial explains how to search the NASA CSDA SpatioTemporal Asset Catalog (STAC) for imagery covering one or more evaluation sites. It is adapted from `csda_stac_search_multiple.ipynb` and intentionally stops after the search results and their footprints have been written to disk.

The workflow has five conceptual stages:

1. Read geographic areas of interest (AOIs) for the sites.
2. Convert those AOIs into search geometries.
3. Connect to the STAC API and choose collections and dates.
4. Search the catalog, potentially using several processes.
5. Combine the results into a GeoDataFrame and write them out.

A STAC catalog is a standardized index of geospatial assets. Each catalog item usually describes one acquisition, including its footprint, date, collection, and links to files or other assets. A STAC search returns metadata about items; it does not automatically download the imagery.

## Before you begin

This notebook assumes that your Python environment already contains the project libraries used by the original workflow, including `geopandas`, `pandas`, `numpy`, `matplotlib`, `pystac-client`, `tqdm`, and the local modules `staclib`, `csdalib`, and `siteslib`.

The local module paths below are examples from the original notebook. Replace them with the paths used by your own environment.

In [ ]:
import sys
from datetime import datetime
from functools import partial
from multiprocessing import Pool, cpu_count
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.append('/projects/code/csda_summaries/lib')
sys.path.append('/projects/code/geoscitools')
sys.path.append('/home/pmontesa/code/csda_summaries/lib')
sys.path.append('/home/pmontesa/code/geoscitools')

import csdalib
import siteslib
import staclib

## 1. Read the evaluation-site AOIs

An area of interest, or AOI, is the geographic area used in a search. Here the AOIs are stored as a GeoJSON file. A GeoDataFrame combines a tabular structure with a geometry column, allowing site attributes and spatial operations to be handled together.

The original workflow reads the evaluation-site file directly from GitHub. Reading a version-controlled source makes the analysis easier to reproduce because the source URL identifies the input dataset.

In [ ]:
RAW_BASE = 'https://raw.githubusercontent.com/pahbs/csda_summaries/master'
SITES_URL = f'{RAW_BASE}/sites/csda_sites_aoi.geojson'
SITES_LABEL = 'eval'

sites_gdf = gpd.read_file(SITES_URL)
print(f'{len(sites_gdf)} site AOIs were read.')
print('Available fields:')
print(list(sites_gdf.columns))
display(sites_gdf.head())

### Inspect the site table before searching

Inspection is a basic quality-control step. Confirm that the site-name field exists, that the geometry column is populated, and that the coordinate reference system is defined. The coordinate reference system (CRS) describes how coordinates relate to locations on Earth.

In [ ]:
SITE_NAME_FIELD = 'Site Name'

if SITE_NAME_FIELD not in sites_gdf.columns:
    raise KeyError(f'Missing {SITE_NAME_FIELD!r}; available fields: {list(sites_gdf.columns)}')
if sites_gdf.geometry.isna().any():
    raise ValueError('At least one site has missing geometry.')

print(f'CRS: {sites_gdf.crs}')
print(f'Unique site names: {sites_gdf[SITE_NAME_FIELD].nunique()}')
display(sites_gdf[[SITE_NAME_FIELD, 'geometry']].head())

## 2. Choose the sites and prepare map geometries

The original notebook supports several search modes. You can search all evaluation sites, select one site, or use a separate site file. The variables below make that choice explicit.

A buffer expands a geometry outward by a specified distance. The search buffer and the display buffer are kept separate because a map may need more surrounding context than the actual STAC query.

In [ ]:
DO_SITE_SUBSET = False
SITE_NAME = 'La Crau'
BUFFER_KM_FOR_SEARCH = 0
BUFFER_KM_FOR_DISPLAY = 0

if DO_SITE_SUBSET:
    sites_gdf = sites_gdf.loc[sites_gdf[SITE_NAME_FIELD] == SITE_NAME].copy()
    if sites_gdf.empty:
        raise ValueError(f'No AOI found for site {SITE_NAME!r}.')
    SITES_LABEL = SITE_NAME.replace(' ', '.')

if sites_gdf.crs is None:
    raise ValueError('The site GeoDataFrame has no CRS.')

# Web Mercator uses metres, so it is convenient for a simple map buffer.
sites_gdf_buffered = sites_gdf.to_crs(3857).copy()
sites_gdf_buffered['geometry'] = sites_gdf_buffered.geometry.buffer(BUFFER_KM_FOR_SEARCH * 1000)
sites_gdf_buffered = sites_gdf_buffered.to_crs(4326)
sites_gdf_buffered = sites_gdf_buffered.loc[~sites_gdf_buffered.geometry.is_empty].copy()

sites_gdf_display = sites_gdf.to_crs(3857).copy()
sites_gdf_display['geometry'] = sites_gdf_display.geometry.buffer(BUFFER_KM_FOR_DISPLAY * 1000)
sites_gdf_display = sites_gdf_display.to_crs(4326)
display(sites_gdf_display[[SITE_NAME_FIELD, 'geometry']])

## 3. Create search regions

Many sites can produce a large number of STAC requests. The helper function `create_global_grid` divides the globe into rectangular cells. Searching by cells reduces the number of very large requests and makes a global search practical.

For a site-only search, the project helper `create_site_dict_from_gdf` converts each AOI into the dictionary format expected by `stac_search_site`. Each dictionary entry contains a name and the geometry information needed by the search function.

In [ ]:
def create_global_grid(grid_size_degrees=30):
    """Create rectangular longitude-latitude search cells."""
    site_dicts = {}
    lons = range(-180, 180, grid_size_degrees)
    lats = range(-90, 90, grid_size_degrees)

    for i, lon_start in enumerate(lons):
        for j, lat_start in enumerate(lats):
            lon_end = min(lon_start + grid_size_degrees, 180)
            lat_end = min(lat_start + grid_size_degrees, 90)
            grid_name = f'grid_{i:02d}_{j:02d}'
            site_dicts[grid_name] = {
                'bbox': [lon_start, lat_start, lon_end, lat_end],
                'lon': (lon_start + lon_end) / 2,
                'lat': (lat_start + lat_end) / 2,
                'buffer': None,
                'geom_type': 'Polygon',
            }
    return site_dicts

DO_GLOBAL_SEARCH = True
if DO_GLOBAL_SEARCH:
    site_dicts = create_global_grid(grid_size_degrees=10)
    print(f'Global search divided into {len(site_dicts)} cells.')
else:
    site_dicts = staclib.create_site_dict_from_gdf(
        sites_gdf_buffered,
        site_name_field=SITE_NAME_FIELD,
        buffer_degrees=0,
    )
    print(f'Site search contains {len(site_dicts)} AOIs.')

CSDA_SEARCH_REGIONS = list(site_dicts.items())
display(CSDA_SEARCH_REGIONS[:2])

## 4. Connect to the CSDA STAC API

The STAC endpoint is a web API. Opening the catalog tests that the endpoint can be reached and that it responds as a STAC catalog. The small test search is useful because a successful network connection does not guarantee that search requests are configured correctly.

In [ ]:
import pystac_client

STAC_URL = 'https://csdap.earthdata.nasa.gov/stac'
catalog = pystac_client.Client.open(STAC_URL)
print(f'Connected to catalog: {catalog.id}')

test_items = list(catalog.search(max_items=1).items())
print(f'Test search returned {len(test_items)} item(s).')

## 5. Define the search query

A STAC query is defined by several filters:

- `collections` limits the search to selected data providers or products.
- `start_date` and `end_date` define the time interval. The end date is written as the start of the following interval in this example.
- `months` provides an additional month filter through the project helper.

The exact collection IDs must be checked against the catalog. A collection name that is visually similar but not identical will return no results or an API error.

In [ ]:
TODAY = datetime.now().strftime('%Y-%m-%d')
START_OF_COMMERCIAL_ERA = '1999-01-01'

COLLECTION_LIST = ['planet', 'airbus-optical', 'satellogic', 'maxar-legion']
YEAR = 2019
START_DATE = f'{YEAR}-01-01'
END_DATE = f'{YEAR + 1}-01-01'
MONTH_LIST = [6, 7, 8]

print('Collections:', COLLECTION_LIST)
print('Date interval:', START_DATE, 'through', END_DATE)
print('Months:', MONTH_LIST)

## 6. Search the catalog for every region

`staclib.stac_search_site` performs the project-specific search and returns a tuple containing a site-level GeoDataFrame, STAC items, and a status message.

The `partial` function fixes the query parameters while leaving the search region as the argument that changes on each call. The pool then distributes those independent searches across worker processes. Start with one process when debugging; parallel requests are faster for large searches but can place more load on the remote service.

In [ ]:
search_func = partial(
    staclib.stac_search_site,
    end_date=END_DATE,
    start_date=START_DATE,
    months=MONTH_LIST,
    collections=COLLECTION_LIST,
)

USE_MULTIPROCESSING = True
n_processes = max(1, cpu_count() - 2) if USE_MULTIPROCESSING else 1
print(f'Using {n_processes} process(es) for {len(CSDA_SEARCH_REGIONS)} search regions.')

if n_processes == 1:
    results = [search_func(region) for region in tqdm(CSDA_SEARCH_REGIONS)]
else:
    with Pool(processes=n_processes) as pool:
        results = list(tqdm(pool.imap(search_func, CSDA_SEARCH_REGIONS), total=len(CSDA_SEARCH_REGIONS)))

stac_gdf_list = []
stac_items_list = []
print('Results by search region:')
for region_name, gdf, items, message in results:
    print(f'  {region_name}: {message}')
    if gdf is not None and not gdf.empty:
        stac_gdf_list.append(gdf)
        stac_items_list.extend(items)

if stac_gdf_list:
    stac_gdf = gpd.GeoDataFrame(pd.concat(stac_gdf_list, ignore_index=True))
    print(f'Total: {len(stac_gdf)} items.')
    if 'site_name' in stac_gdf.columns:
        print(f"Sites represented: {stac_gdf['site_name'].nunique()}")
        display(stac_gdf['site_name'].value_counts())
else:
    stac_gdf = gpd.GeoDataFrame()
    print('No STAC items were found.')

## 7. Write the STAC items and footprint table

The search produces two useful representations of the result:

- `stac_items_list` contains STAC item objects and preserves the catalog metadata structure.
- `stac_gdf` is a tabular spatial representation in which each row is an item and the geometry is its footprint.

Asset fields can contain lists or nested objects that are inconvenient in a GeoPackage table. The final cleanup removes columns whose names contain `asset`, converts remaining list-valued fields to strings, and writes the result as a GeoPackage. The original STAC item objects are also written to JSON.

In [ ]:
from pystac import ItemCollection

if not stac_gdf.empty:
    collection_label = '_'.join(COLLECTION_LIST)
    output_stac_json = Path(f'stac_items_{SITES_LABEL}_{collection_label}_{YEAR}.json')
    output_footprints = Path(f'stac_{SITES_LABEL}_{collection_label}_{YEAR}.gpkg')

    ItemCollection(stac_items_list).save_object(dest_href=str(output_stac_json))

    stac_gdf_to_write = stac_gdf.drop(
        columns=[column for column in stac_gdf.columns if 'asset' in column.lower()],
        errors='ignore',
    ).copy()
    for column in stac_gdf_to_write.columns:
        if stac_gdf_to_write[column].map(lambda value: isinstance(value, list)).any():
            stac_gdf_to_write[column] = stac_gdf_to_write[column].map(
                lambda value: str(value) if isinstance(value, list) else value
            )

    stac_gdf_to_write.to_file(output_footprints, driver='GPKG')
    print(f'Wrote STAC items to {output_stac_json}')
    print(f'Wrote footprint GeoPackage to {output_footprints}')
else:
    print('Nothing was written because stac_gdf is empty.')

## What has been produced?

At this point the workflow has completed the catalog-search stage. The JSON file preserves the STAC item records, while the GeoPackage provides a GIS-friendly footprint table for inspection in software such as QGIS. Neither file contains downloaded image pixels.

Before continuing to image download or scientific analysis, inspect the number of returned items, their acquisition dates, collection names, spatial footprints, and any duplicate acquisitions. Those checks determine whether the search query represents the intended population of observations.